# Flight Prices: Exploratory Data Analysis (EDA)

**Project:** When should you buy a plane ticket? Predicting U.S. domestic flight prices and price drops.\
**Course:** Intro to Machine Learning, Fall 2026\
**Author:** Anita Roese

**Purpose:** This notebook explores and analyzes the `selected_routes_labeled.csv` file created in `02_route_extraction`: every search for the six selected routes, with the `wait` label. It covers a data quality assessment, univariate analysis, target variable analysis, bivariate and multivariate analysis, and other special checks. Both targets are studied: `totalFare`, which the linear regression will predict, and `wait`, which the logistic regression will classify.

## 1. Setup

In [2]:
import os
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.max_columns", None)

In [4]:
labeled_path = "data/selected_routes_labeled.csv"   # created in 02_route_extraction
fig_dir = "figures"                                 # folder for the figures used in the EDA report

SAMPLE_SIZE = 200_000   # random rows used for plots and slow statistical tests
RANDOM_STATE = 42       # makes the sample reproducible

os.makedirs(fig_dir, exist_ok=True)   # create the figures folder if it doesn't exist

## 2. Dataset structure

The labeled file is loaded with DuckDB, which detects dates and True/False columns automatically, and converted to a pandas DataFrame. As in notebook 01, `daysUntilDeparture` and `travelMinutes` are engineered, and a `route` column combines origin and destination.

In [5]:
df = duckdb.sql(f"SELECT * FROM '{labeled_path}'").df()   # .df() returns a pandas DataFrame

df["searchDate"] = pd.to_datetime(df["searchDate"])
df["flightDate"] = pd.to_datetime(df["flightDate"])
df["daysUntilDeparture"] = (df["flightDate"] - df["searchDate"]).dt.days

days  = df["travelDuration"].str.extract(r"(\d+)D")[0].astype(float).fillna(0)
hours = df["travelDuration"].str.extract(r"(\d+)H")[0].astype(float).fillna(0)
mins  = df["travelDuration"].str.extract(r"(\d+)M")[0].astype(float).fillna(0)
df["travelMinutes"] = days * 1440 + hours * 60 + mins

df["route"] = df["startingAirport"] + "→" + df["destinationAirport"]

print("Shape:", df.shape)

Shape: (3339506, 16)


In [6]:
# Variable dictionary: data type and number of unique values of each column
pd.DataFrame({"dtype": df.dtypes.astype(str), "unique": df.nunique()})

,dtype,unique
legId,str,207200
searchDate,datetime64[us],164
flightDate,datetime64[us],213
startingAirport,str,6
destinationAirport,str,4
totalFare,float64,12052
seatsRemaining,int64,11
travelDuration,str,1084
totalTravelDistance,Int64,595
isNonStop,bool,2


In [7]:
df.head()

,legId,searchDate,flightDate,startingAirport,destinationAirport,totalFare,seatsRemaining,travelDuration,totalTravelDistance,isNonStop,isBasicEconomy,minFareNext7Days,wait,daysUntilDeparture,travelMinutes,route
0,6dda226b795def883f7675fc7d132ab1,2022-04-23,2022-04-29,BOS,LGA,118.6,4,PT1H24M,185,True,False,103.60,1,6,84.0,BOS→LGA
1,6dda226b795def883f7675fc7d132ab1,2022-04-24,2022-04-29,BOS,LGA,118.6,4,PT1H24M,185,True,False,103.60,1,5,84.0,BOS→LGA
2,6dda226b795def883f7675fc7d132ab1,2022-04-25,2022-04-29,BOS,LGA,103.6,1,PT1H24M,185,True,False,103.60,0,4,84.0,BOS→LGA
3,6dda226b795def883f7675fc7d132ab1,2022-04-26,2022-04-29,BOS,LGA,118.6,2,PT1H24M,185,True,False,103.60,1,3,84.0,BOS→LGA
4,6dda226b795def883f7675fc7d132ab1,2022-04-27,2022-04-29,BOS,LGA,103.6,7,PT1H24M,185,True,False,168.61,0,2,84.0,BOS→LGA


*Interpretation:* All data types are correct after loading: `searchDate` and `flightDate` are dates, and `travelDuration`, which is stored as text in ISO 8601 format, was converted to the numeric feature `travelMinutes` (both have 1,084 unique values, so the conversion is one-to-one). `wait` is stored as an integer but is a binary categorical target (0 = buy now, 1 = wait), and `route`, `startingAirport`, and `destinationAirport` are categorical variables with only 6, 6, and 4 levels, so they will need encoding before modeling. `legId` is an identifier with 207,200 unique values and is used only to group searches of the same flight, never as a feature. Finally, `searchDate` has 164 unique values while the period from April 16 to September 28 has 166 days, so two days have no searches at all.

## Data quality checks
### 3.1 Missing values

In [8]:
missing = pd.DataFrame({"missing": df.isna().sum(), "pct": (df.isna().mean() * 100).round(2)})
missing[missing["missing"] > 0]

,missing,pct
totalTravelDistance,319168,9.56
minFareNext7Days,244334,7.32


In [9]:
miss = df["totalTravelDistance"].isna()
print("Missing % by route:")
print((miss.groupby(df["route"]).mean() * 100).round(2))
print("\nMissing % by isNonStop:")
print((miss.groupby(df["isNonStop"]).mean() * 100).round(2))

Missing % by route:
route
ATL→LAX    22.65
BOS→LGA     0.51
CLT→ORD     4.44
DFW→LAX    14.62
LAX→BOS     4.82
ORD→LGA     5.00
Name: totalTravelDistance, dtype: float64

Missing % by isNonStop:
isNonStop
False    12.92
True      4.09
Name: totalTravelDistance, dtype: float64


In [10]:
no_future = df["minFareNext7Days"].isna()
departs_soon = df["daysUntilDeparture"] <= 7
print(f"No later search within 7 days: {no_future.mean():.2%} of rows")
print(f"  ...and departs within 7 days (expected): {(no_future & departs_soon).mean():.2%}")
print(f"  ...but departs later (unobserved):       {(no_future & ~departs_soon).mean():.2%}")

No later search within 7 days: 7.32% of rows
  ...and departs within 7 days (expected): 3.12%
  ...but departs later (unobserved):       4.19%


*Interpretation:* Two columns have missing values. `totalTravelDistance` is missing for 9.56% of rows, and the rate depends on other columns: from 0.51% (BOS→LGA) to 22.65% (ATL→LAX), and 12.92% for connecting flights versus 4.09% for nonstop flights. The values are therefore **missing at random (MAR)**, not completely at random; the likely cause is that total distance is often not reported for connecting itineraries. Because distance is nearly fixed for a given route and type of trip, missing values will be imputed with the median distance of the same route and nonstop status (the correlation analysis in Section 6 will show whether distance is redundant with `travelMinutes`).

`minFareNext7Days` is empty for 7.32% of rows by design, when the flight has no later search within 7 days. For 3.12% of rows this is because the flight departs within 7 days, so `wait = 0` is correct. For the other 4.19%, the flight departs later but was not observed again, so a price drop may have been missed; this limits the reliability of about 4% of the labels and means the share of "wait" labels is a lower bound. `minFareNext7Days` is used only to build the label and is never a feature.

### 3.2 Duplicates and cardinality

In [11]:
print("Fully duplicated rows:", df.duplicated().sum())
print("Rows with the same legId and searchDate:", df.duplicated(subset=["legId", "searchDate"]).sum())
print()
print("Unique legId:", df["legId"].nunique(), "of", len(df), "rows")
print()
# Share of the most common value in each column (near-constant columns are close to 100%)
top_share = {c: df[c].value_counts(normalize=True, dropna=False).iloc[0] * 100 for c in df.columns}
pd.Series(top_share, name="% most common value").round(2).sort_values(ascending=False)

Fully duplicated rows: 0
Rows with the same legId and searchDate: 0

Unique legId: 207200 of 3339506 rows



isBasicEconomy         83.60
wait                   65.10
isNonStop              61.93
seatsRemaining         42.37
destinationAirport     37.98
route                  20.36
startingAirport        20.36
totalTravelDistance    11.24
minFareNext7Days        7.32
travelDuration          2.17
travelMinutes           2.17
daysUntilDeparture      2.11
totalFare               1.76
flightDate              0.89
searchDate              0.81
legId                   0.00
Name: % most common value, dtype: float64

*Interpretation:* There are no duplicated rows, and no flight appears twice on the same search date, so each flight has exactly one observed price per day and the `wait` label is unambiguous. `legId` is an identifier (207,200 flights, about 16 searches each) and is used only for grouping. No column is constant or near-constant: the most imbalanced is `isBasicEconomy` (83.6% False). The categorical variables have few levels (at most 6), so there are no high-cardinality features. One value of `seatsRemaining` covers 42.4% of rows, which is checked below as a possible cap.

### 3.3 Validity checks

In [12]:
checks = {
    "totalFare <= 0": (df["totalFare"] <= 0).sum(),
    "totalFare < 50 USD": (df["totalFare"] < 50).sum(),
    "seatsRemaining = 0": (df["seatsRemaining"] == 0).sum(),
    f"seatsRemaining at max ({df['seatsRemaining'].max()})": (df["seatsRemaining"] == df["seatsRemaining"].max()).sum(),
    "travelMinutes < 30": (df["travelMinutes"] < 30).sum(),
    "travelMinutes > 24 h": (df["travelMinutes"] > 1440).sum(),
    "flightDate before searchDate": (df["flightDate"] < df["searchDate"]).sum(),
    "daysUntilDeparture = 0": (df["daysUntilDeparture"] == 0).sum(),
}
pd.Series(checks, name="rows")

totalFare <= 0                       0
totalFare < 50 USD               10020
seatsRemaining = 0              298422
seatsRemaining at max (10)       11410
travelMinutes < 30                   0
travelMinutes > 24 h               315
flightDate before searchDate         0
daysUntilDeparture = 0               0
Name: rows, dtype: int64

In [15]:
(df["seatsRemaining"].value_counts(normalize=True).sort_index() * 100).round(2)

seatsRemaining
0      8.94
1      5.05
2      4.79
3      4.44
4      4.47
5      3.43
6      3.18
7     42.37
8      1.21
9     21.79
10     0.34
Name: proportion, dtype: float64

In [13]:
# Distance should be almost fixed for a route: compare min, median and max per route
df.groupby("route")["totalTravelDistance"].agg(["min", "median", "max"])

,min,median,max
route,,,
ATL→LAX,1389,2013.0,3220
BOS→LGA,185,185.0,1856
CLT→ORD,592,921.0,2249
DFW→LAX,474,1456.0,2668
LAX→BOS,339,2643.0,3599
ORD→LGA,720,720.0,2296


In [14]:
# The 5 cheapest tickets, to judge whether low fares are plausible
df.nsmallest(5, "totalFare")[["route", "searchDate", "flightDate", "totalFare", "isBasicEconomy", "isNonStop"]]

,route,searchDate,flightDate,totalFare,isBasicEconomy,isNonStop
330035,DFW→LAX,2022-08-11,2022-08-30,41.59,False,True
330036,DFW→LAX,2022-08-12,2022-08-30,41.59,False,True
330037,DFW→LAX,2022-08-13,2022-08-30,41.59,False,True
330038,DFW→LAX,2022-08-14,2022-08-30,41.59,False,True
330039,DFW→LAX,2022-08-15,2022-08-30,41.59,False,True


*Interpretation:*
- **Inconsistent labels:** none; airport codes and True/False values are consistent.
- **Incorrect types:** fixed during loading (dates parsed, `travelDuration` converted to minutes).
- **Range / plausibility:** no zero or negative fares. The 10,020 fares under 50 USD (0.3%) look genuine; the cheapest (41.59 USD, DFW→LAX nonstop) stays stable across consecutive searches, consistent with ultra-low-cost airlines. 315 trips (0.01%) last more than 24 hours because of long connections and are kept. However, `totalTravelDistance` contains impossible values: on several routes the minimum is far below the direct distance (e.g., 339 miles for LAX→BOS, which is about 2,600 miles direct), likely because only some segments of a connecting trip were added up.
- **Invalid dates:** none; every flight date is after its search date (1 to 60 days).
- **Caps / floors:** `seatsRemaining` is censored. 42.4% of rows show 7 and 21.8% show 9, while values 1–6 each cover only 3–5%, so 7 and 9 most likely mean "7 or more" and "9 or more". The value 0 (8.9%) most likely means the seat count was not reported, since a ticket for sale cannot have zero seats.

**Verdict:** the fares, dates, and labels are reliable. Two features need treatment before modeling: `totalTravelDistance` (missing and partly invalid, and largely determined by the route) is a candidate for removal, and `seatsRemaining` should be converted into flags (e.g., few seats left = 1–3, seats unknown = 0) instead of being used as a count.